# MCP Server

Starts the FastMCP server that provides agent card discovery (via embeddings), travel data queries, and Google Places lookup.

In [1]:
import json
import os
import sqlite3
from pathlib import Path

import numpy as np
import requests
from openai import OpenAI
from dotenv import load_dotenv
from mcp.server.fastmcp import FastMCP

load_dotenv()

AGENT_CARDS_DIR = 'agent_cards'
EMBEDDING_MODEL = 'text-embedding-3-small'
SQLITE_DB = 'travel_agency.db'
PLACES_API_URL = 'https://places.googleapis.com/v1/places:searchText'
HOST, PORT = 'localhost', 10100

oai = OpenAI()

In [2]:
def load_agent_cards():
    card_uris, agent_cards = [], []
    for filename in sorted(os.listdir(AGENT_CARDS_DIR)):
        if filename.endswith('.json'):
            with open(Path(AGENT_CARDS_DIR) / filename) as f:
                data = json.load(f)
            card_uris.append(f'resource://agent_cards/{Path(filename).stem}')
            agent_cards.append(data)
    return card_uris, agent_cards


def build_agent_card_embeddings():
    card_uris, agent_cards = load_agent_cards()
    texts = [json.dumps(card) for card in agent_cards]
    result = oai.embeddings.create(input=texts, model=EMBEDDING_MODEL)
    embeddings = [item.embedding for item in result.data]
    print(f'Loaded {len(card_uris)} agent cards with embeddings')
    return card_uris, agent_cards, np.array(embeddings)


card_uris, agent_cards, card_embeddings = build_agent_card_embeddings()

Loaded 5 agent cards with embeddings


In [3]:
mcp = FastMCP('agent-cards', host=HOST, port=PORT)


@mcp.tool(name='find_agent', description='Find the most relevant agent card for a query.')
def find_agent(query: str) -> str:
    result = oai.embeddings.create(input=[query], model=EMBEDDING_MODEL)
    query_embedding = result.data[0].embedding
    scores = card_embeddings @ query_embedding
    return agent_cards[int(np.argmax(scores))]


@mcp.tool()
def query_places_data(query: str):
    """Query Google Places."""
    api_key = os.getenv('GOOGLE_PLACES_API_KEY')
    if not api_key:
        return {'places': []}
    headers = {
        'X-Goog-Api-Key': api_key,
        'X-Goog-FieldMask': 'places.id,places.displayName,places.formattedAddress',
        'Content-Type': 'application/json',
    }
    try:
        resp = requests.post(PLACES_API_URL, headers=headers, json={'textQuery': query, 'languageCode': 'en', 'maxResultCount': 10})
        resp.raise_for_status()
        return resp.json()
    except Exception:
        return {'places': []}


@mcp.tool()
def query_travel_data(query: str) -> dict:
    """Query the travel SQLite database (flights, hotels, rental_cars). Only SELECT queries."""
    if not query or not query.strip().upper().startswith('SELECT'):
        raise ValueError(f'Invalid query: {query}')
    try:
        with sqlite3.connect(SQLITE_DB) as conn:
            conn.row_factory = sqlite3.Row
            rows = conn.cursor().execute(query).fetchall()
            return json.dumps({'results': [dict(row) for row in rows]})
    except Exception as e:
        return {'error': str(e)}


@mcp.resource('resource://agent_cards/list', mime_type='application/json')
def get_agent_cards() -> dict:
    return {'agent_cards': list(card_uris)}


@mcp.resource('resource://agent_cards/{card_name}', mime_type='application/json')
def get_agent_card(card_name: str) -> dict:
    uri = f'resource://agent_cards/{card_name}'
    matches = [card for u, card in zip(card_uris, agent_cards) if u == uri]
    return {'agent_card': matches}

In [ ]:
print(f'MCP Server running at http://{HOST}:{PORT}')
await mcp.run_sse_async()

INFO:     Started server process [3731]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://localhost:10100 (Press CTRL+C to quit)


MCP Server running at http://localhost:10100
INFO:     ::1:50342 - "GET /sse HTTP/1.1" 200 OK
INFO:     ::1:50343 - "GET /sse HTTP/1.1" 200 OK
INFO:     ::1:50344 - "GET /sse HTTP/1.1" 200 OK
INFO:     ::1:50345 - "GET /sse HTTP/1.1" 200 OK
INFO:     ::1:50374 - "POST /sse HTTP/1.1" 405 Method Not Allowed
INFO:     ::1:50375 - "GET /sse HTTP/1.1" 200 OK
INFO:     ::1:50376 - "POST /messages/?session_id=5b038d9c91a944198b27b4d19c1bc80d HTTP/1.1" 202 Accepted
INFO:     ::1:50376 - "POST /messages/?session_id=5b038d9c91a944198b27b4d19c1bc80d HTTP/1.1" 202 Accepted
INFO:     ::1:50376 - "POST /messages/?session_id=5b038d9c91a944198b27b4d19c1bc80d HTTP/1.1" 202 Accepted


[04/05/26 21:10:38] INFO     Processing request of type ListToolsRequest                              ]8;id=218006;file:///Users/paul/Documents/code/a2a-test/.venv/lib/python3.13/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=753807;file:///Users/paul/Documents/code/a2a-test/.venv/lib/python3.13/site-packages/mcp/server/lowlevel/server.py#720\720]8;;\

INFO:     ::1:50377 - "POST /messages/?session_id=5b038d9c91a944198b27b4d19c1bc80d HTTP/1.1" 202 Accepted
INFO:     ::1:50378 - "POST /messages/?session_id=5b038d9c91a944198b27b4d19c1bc80d HTTP/1.1" 202 Accepted


                    INFO     Processing request of type ListResourcesRequest                          ]8;id=992079;file:///Users/paul/Documents/code/a2a-test/.venv/lib/python3.13/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=990728;file:///Users/paul/Documents/code/a2a-test/.venv/lib/python3.13/site-packages/mcp/server/lowlevel/server.py#720\720]8;;\

                    INFO     Processing request of type ListPromptsRequest                            ]8;id=825767;file:///Users/paul/Documents/code/a2a-test/.venv/lib/python3.13/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=450941;file:///Users/paul/Documents/code/a2a-test/.venv/lib/python3.13/site-packages/mcp/server/lowlevel/server.py#720\720]8;;\